# Building Segmentation Training — Kaggle

This notebook trains the building-segmentation model used after the OpenSR super-resolution stage.

## Pipeline

```text
30 OpenSR GeoTIFFs
        ↓
Read RGB bands: B04 / B03 / B02
        ↓
Download OSM building footprints
        ↓
Rasterize footprints on the SR image grid
        ↓
Check image / mask alignment
        ↓
Image-level train / validation split
        ↓
Balanced 256×256 patches
        ↓
Data augmentation
        ↓
U-Net training
        ↓
Validation threshold selection
        ↓
unet_buildings.keras
+
segmentation_training_config.json
```

## Input

The Kaggle dataset is already extracted by Kaggle and displayed as **sr dataset**.

The notebook automatically searches `/kaggle/input` for the folder containing the SR `.tif` files, so you normally do not need to hard-code the Kaggle dataset slug.

Expected OpenSR GeoTIFF band order:

- Band 1 = B04 = Red
- Band 2 = B03 = Green
- Band 3 = B02 = Blue
- Band 4 = B08 = NIR

This segmentation model currently uses **RGB only: bands 1, 2 and 3**.

## Kaggle settings

- **Accelerator:** GPU
- **Internet:** ON, because OpenStreetMap footprints are downloaded during label generation.

In [ ]:
# ============================================================
# 0. INSTALL DEPENDENCIES
# ============================================================

!pip install -q osmnx

In [ ]:
# ============================================================
# 1. IMPORTS + GPU CHECK
# ============================================================

import os
import glob
import json
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf

import rasterio
from rasterio.features import rasterize

import geopandas as gpd
import osmnx as ox

from shapely.geometry import box
from pyproj import Transformer

warnings.filterwarnings("ignore", category=UserWarning)

print("TensorFlow :", tf.__version__)
print("Rasterio   :", rasterio.__version__)
print("GeoPandas  :", gpd.__version__)
print("OSMnx      :", ox.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
else:
    print("⚠️ No GPU detected. Training will be much slower.")

## 2. Locate the extracted `sr dataset`

Kaggle may convert the display name `sr dataset` into a path such as:

```text
/kaggle/input/sr-dataset
```

The next cell avoids relying on the exact slug. It scans `/kaggle/input` and selects the directory containing the largest number of `.tif` files.

In [ ]:
# ============================================================
# 2. AUTO-DETECT THE EXTRACTED SR DATASET
# ============================================================

KAGGLE_INPUT = "/kaggle/input"

all_tifs_under_input = sorted(
    glob.glob(
        os.path.join(KAGGLE_INPUT, "**", "*.tif"),
        recursive=True
    )
)

if not all_tifs_under_input:
    raise FileNotFoundError(
        "No .tif files were found under /kaggle/input. "
        "Attach the 'sr dataset' Kaggle dataset first."
    )

# Count TIFFs by first-level Kaggle dataset folder.
dataset_groups = {}

for tif_path in all_tifs_under_input:
    rel = os.path.relpath(tif_path, KAGGLE_INPUT)
    first_folder = rel.split(os.sep)[0]

    dataset_groups.setdefault(first_folder, [])
    dataset_groups[first_folder].append(tif_path)

print("Detected Kaggle input datasets containing GeoTIFFs:\n")

for name, paths in sorted(
    dataset_groups.items(),
    key=lambda x: len(x[1]),
    reverse=True
):
    print(f"  {name:40s} -> {len(paths)} TIFF(s)")

# Prefer names resembling "sr dataset", otherwise choose the group
# containing the most TIFFs.
preferred = [
    name for name in dataset_groups
    if ("sr" in name.lower() and "dataset" in name.lower())
]

if preferred:
    dataset_name = max(
        preferred,
        key=lambda n: len(dataset_groups[n])
    )
else:
    dataset_name = max(
        dataset_groups,
        key=lambda n: len(dataset_groups[n])
    )

DATASET_PATH = os.path.join(
    KAGGLE_INPUT,
    dataset_name
)

all_tif_files = sorted(
    glob.glob(
        os.path.join(DATASET_PATH, "**", "*.tif"),
        recursive=True
    )
)

print("\n✅ Selected dataset:")
print(DATASET_PATH)

print(f"\n✅ SR GeoTIFFs found: {len(all_tif_files)}")

for path in all_tif_files[:10]:
    print(" -", path)

if len(all_tif_files) != 30:
    print(
        f"\n⚠️ Expected about 30 SR images, "
        f"but found {len(all_tif_files)}. "
        "Training can continue, but verify that the correct dataset was selected."
    )

# Try to locate the manifest generated by the SR notebook.
manifest_candidates = sorted(
    glob.glob(
        os.path.join(DATASET_PATH, "**", "manifest.csv"),
        recursive=True
    )
)

MANIFEST_PATH = (
    manifest_candidates[0]
    if manifest_candidates
    else None
)

if MANIFEST_PATH:
    print("\n✅ Manifest:")
    print(MANIFEST_PATH)
else:
    print("\nℹ️ manifest.csv not found. This is not required for training.")

## 3. Training configuration

In [ ]:
# ============================================================
# 3. CONFIGURATION
# ============================================================

OUTPUT_DIR = "/kaggle/working"
CACHE_DIR = os.path.join(
    OUTPUT_DIR,
    "footprint_cache"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ------------------------------------------------------------
# INPUT BANDS
# ------------------------------------------------------------

# OpenSR output:
# 1 = B04 Red
# 2 = B03 Green
# 3 = B02 Blue
# 4 = B08 NIR
#
# Current U-Net input = RGB.
BANDS = [1, 2, 3]

# ------------------------------------------------------------
# LABEL GENERATION
# ------------------------------------------------------------

# Small geographic buffer around the image bounds when requesting OSM.
BUFFER_DEG = 0.002

RASTERIZE_ALL_TOUCHED = False

# Usually keep 0 initially. Increase only if labels look systematically
# smaller than roofs after visual inspection.
LABEL_DILATE_PIXELS = 0

# ------------------------------------------------------------
# PATCHES
# ------------------------------------------------------------

PATCH_SIZE = 256

# 30 images × 60 patches ~= 1800 patches before filtering.
N_PATCHES_PER_IMAGE = 60

# Fraction of requested patches centered on building pixels.
POS_FRACTION = 0.60

# Minimum building coverage for a positive patch.
MIN_POS_RATIO = 0.01

# Maximum building coverage for a negative patch.
MAX_NEG_RATIO = 0.001

# ------------------------------------------------------------
# SPLIT
# ------------------------------------------------------------

# IMPORTANT: images are split BEFORE patch generation.
VAL_IMAGE_FRACTION = 0.20

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-4
SEED = 42

MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "unet_buildings.keras"
)

CONFIG_PATH = os.path.join(
    OUTPUT_DIR,
    "segmentation_training_config.json"
)

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuration ready.")

In [ ]:
# ============================================================
# 4. OPTIONAL: INSPECT THE SR DATASET MANIFEST
# ============================================================

if MANIFEST_PATH is not None:

    sr_manifest = pd.read_csv(MANIFEST_PATH)

    print("Manifest rows:", len(sr_manifest))

    display(sr_manifest.head(30))

    if "category" in sr_manifest.columns:
        print("\nCategory balance:")
        display(
            sr_manifest["category"]
            .value_counts()
            .rename("count")
            .to_frame()
        )

else:
    print("No manifest available. Continuing with GeoTIFFs only.")

## 5. Read and validate the OpenSR GeoTIFFs

For each image we verify:

- at least 3 bands are present;
- a CRS exists;
- the geotransform exists;
- RGB can be read successfully.

The network receives a robust **2nd–98th percentile normalization per band**. The exact same preprocessing will later be reused during inference.

In [ ]:
# ============================================================
# 5. GEOTIFF READING + NORMALIZATION
# ============================================================

def percentile_normalize(
    img,
    pmin=2,
    pmax=98,
    eps=1e-6
):
    img = img.astype(np.float32)

    out = np.zeros_like(
        img,
        dtype=np.float32
    )

    for c in range(img.shape[2]):

        band = img[..., c]

        finite = np.isfinite(band)

        if not finite.any():
            continue

        lo = np.percentile(
            band[finite],
            pmin
        )

        hi = np.percentile(
            band[finite],
            pmax
        )

        if hi - lo < eps:
            out[..., c] = 0.0
        else:
            out[..., c] = (
                band - lo
            ) / (
                hi - lo
            )

    out = np.nan_to_num(
        out,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(
        out,
        0.0,
        1.0
    )


def load_geotiff_rgb(
    path,
    bands=BANDS
):
    with rasterio.open(path) as src:

        if src.count < max(bands):
            raise ValueError(
                f"{os.path.basename(path)} has "
                f"{src.count} band(s); requested {bands}."
            )

        if src.crs is None:
            raise ValueError(
                f"{os.path.basename(path)} has no CRS."
            )

        raw = np.stack(
            [
                src.read(b)
                for b in bands
            ],
            axis=-1
        )

        profile = src.profile.copy()
        crs = src.crs
        transform = src.transform
        bounds = src.bounds

    img = percentile_normalize(raw)

    return (
        img,
        profile,
        crs,
        transform,
        bounds
    )


print("Validating SR GeoTIFFs...\n")

valid_tif_files = []

for path in all_tif_files:
    try:
        img, profile, crs, transform, bounds = (
            load_geotiff_rgb(path)
        )

        valid_tif_files.append(path)

        print(
            f"✅ {os.path.basename(path):50s} "
            f"{img.shape} | "
            f"{crs} | "
            f"{abs(transform.a):.2f} m/pixel"
        )

    except Exception as e:
        print(
            f"❌ {os.path.basename(path)} -> {e}"
        )

print(
    f"\nValid images: "
    f"{len(valid_tif_files)}/{len(all_tif_files)}"
)

if len(valid_tif_files) < 2:
    raise RuntimeError(
        "At least two valid images are required."
    )

all_tif_files = valid_tif_files

In [ ]:
# ============================================================
# 6. DISPLAY EVERY TRAINING IMAGE INDIVIDUALLY
# ============================================================

for i, path in enumerate(
    all_tif_files,
    start=1
):

    img, _, crs, transform, bounds = (
        load_geotiff_rgb(path)
    )

    print("\n" + "=" * 75)
    print(f"IMAGE {i}/{len(all_tif_files)}")
    print(os.path.basename(path))
    print("CRS       :", crs)
    print(
        "Resolution:",
        (
            abs(transform.a),
            abs(transform.e)
        )
    )

    plt.figure(figsize=(7, 7))
    plt.imshow(img)
    plt.title(
        os.path.basename(path),
        fontsize=11
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Generate OSM building masks

The GeoTIFF footprint is converted to WGS84 for the OpenStreetMap request. OSM polygons are then reprojected back to the GeoTIFF CRS and rasterized using the image's own transform and dimensions.

In [ ]:
# ============================================================
# 7. OSM LABEL GENERATION
# ============================================================

def bounds_to_wgs84(
    bounds,
    src_crs
):
    transformer = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True
    )

    corners = [
        transformer.transform(
            bounds.left,
            bounds.bottom
        ),
        transformer.transform(
            bounds.left,
            bounds.top
        ),
        transformer.transform(
            bounds.right,
            bounds.bottom
        ),
        transformer.transform(
            bounds.right,
            bounds.top
        ),
    ]

    lons = [x for x, y in corners]
    lats = [y for x, y in corners]

    return (
        min(lons),
        min(lats),
        max(lons),
        max(lats)
    )


def download_osm_buildings(
    bounds,
    src_crs,
    buffer_deg=BUFFER_DEG
):
    west, south, east, north = (
        bounds_to_wgs84(
            bounds,
            src_crs
        )
    )

    request_poly = box(
        west - buffer_deg,
        south - buffer_deg,
        east + buffer_deg,
        north + buffer_deg
    )

    image_poly_src = box(
        bounds.left,
        bounds.bottom,
        bounds.right,
        bounds.top
    )

    gdf = ox.features_from_polygon(
        request_poly,
        tags={"building": True}
    )

    if len(gdf) == 0:
        return gpd.GeoDataFrame(
            {"geometry": []},
            crs=src_crs
        )

    if gdf.crs is None:
        gdf = gdf.set_crs(
            "EPSG:4326"
        )

    gdf = gdf.to_crs(src_crs)

    gdf = gdf[
        gdf.geometry.notnull()
        & gdf.geometry.is_valid
        & gdf.geometry.geom_type.isin(
            [
                "Polygon",
                "MultiPolygon"
            ]
        )
    ].copy()

    # Keep only buildings touching the actual SR image.
    gdf = gdf[
        gdf.intersects(
            image_poly_src
        )
    ].copy()

    return gdf


def rasterize_buildings(
    footprints_gdf,
    out_shape,
    transform
):
    if len(footprints_gdf) == 0:
        return np.zeros(
            out_shape,
            dtype=np.uint8
        )

    shapes = [
        (geom, 1)
        for geom
        in footprints_gdf.geometry
    ]

    mask = rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype=np.uint8,
        all_touched=RASTERIZE_ALL_TOUCHED
    )

    return mask


def footprint_cache_path(
    image_path
):
    # Stable cache name across Python sessions.
    digest = hashlib.md5(
        str(image_path).encode("utf-8")
    ).hexdigest()[:10]

    stem = Path(image_path).stem

    return os.path.join(
        CACHE_DIR,
        f"{stem}_{digest}_footprints.geojson"
    )


images = []
masks = []
metadata = []

print(
    f"Generating labels for "
    f"{len(all_tif_files)} images...\n"
)

for i, path in enumerate(
    all_tif_files,
    start=1
):

    print("\n" + "-" * 70)

    print(
        f"[{i}/{len(all_tif_files)}] "
        f"{os.path.basename(path)}"
    )

    try:

        (
            img,
            profile,
            crs,
            transform,
            bounds
        ) = load_geotiff_rgb(path)

        cache_path = (
            footprint_cache_path(path)
        )

        if os.path.exists(cache_path):

            footprints = (
                gpd.read_file(
                    cache_path
                )
                .to_crs(crs)
            )

            print(
                "Loaded cached OSM footprints:",
                len(footprints)
            )

        else:

            footprints = (
                download_osm_buildings(
                    bounds,
                    crs
                )
            )

            print(
                "OSM footprints found:",
                len(footprints)
            )

            if len(footprints) > 0:
                footprints.to_file(
                    cache_path,
                    driver="GeoJSON"
                )

        mask = rasterize_buildings(
            footprints,
            out_shape=img.shape[:2],
            transform=transform
        )

        if LABEL_DILATE_PIXELS > 0:

            k = (
                2 * LABEL_DILATE_PIXELS
                + 1
            )

            kernel = (
                cv2.getStructuringElement(
                    cv2.MORPH_ELLIPSE,
                    (k, k)
                )
            )

            mask = cv2.dilate(
                mask,
                kernel,
                iterations=1
            )

        coverage = float(
            mask.mean()
        )

        print(
            f"Building pixel coverage: "
            f"{100 * coverage:.2f}%"
        )

        if mask.sum() == 0:
            print(
                "⚠️ Skipped: no OSM "
                "building pixels."
            )
            continue

        images.append(img)
        masks.append(mask)

        metadata.append({
            "name":
                os.path.basename(path),

            "path":
                path,

            "crs":
                str(crs),

            "width":
                int(img.shape[1]),

            "height":
                int(img.shape[0]),

            "building_coverage":
                coverage
        })

    except Exception as e:

        print("❌ ERROR:", e)


print(
    f"\n✅ Usable labelled images: "
    f"{len(images)}"
)

if len(images) < 5:
    raise RuntimeError(
        "Too few labelled images remain. "
        "Inspect OSM/API errors and dataset locations."
    )

## 8. **STOP AND INSPECT THIS CELL BEFORE TRAINING**

The red regions must line up with real roofs. If the overlays are systematically shifted, do **not** train yet.

In [ ]:
# ============================================================
# 8. IMAGE / LABEL ALIGNMENT CHECK
# ============================================================

for i in range(len(images)):

    img = images[i]
    mask = masks[i]
    info = metadata[i]

    print("\n" + "=" * 75)

    print(
        f"LABEL CHECK "
        f"{i + 1}/{len(images)}"
    )

    print(info["name"])

    print(
        f"Building coverage: "
        f"{100 * info['building_coverage']:.2f}%"
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6)
    )

    axes[0].imshow(img)
    axes[0].set_title("SR RGB")

    axes[1].imshow(
        mask,
        cmap="gray"
    )
    axes[1].set_title(
        "OSM binary mask"
    )

    axes[2].imshow(img)
    axes[2].imshow(
        mask,
        cmap="Reds",
        alpha=0.40
    )
    axes[2].set_title(
        "Alignment overlay"
    )

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 9. Image-level train/validation split

We split **whole images first**. This prevents patches from the same city tile appearing in both training and validation.

In [ ]:
# ============================================================
# 9. IMAGE-LEVEL SPLIT
# ============================================================

rng = np.random.default_rng(SEED)

indices = np.arange(
    len(images)
)

rng.shuffle(indices)

n_val_images = max(
    1,
    int(
        round(
            len(images)
            * VAL_IMAGE_FRACTION
        )
    )
)

n_val_images = min(
    n_val_images,
    len(images) - 1
)

val_image_idx = (
    indices[:n_val_images]
)

train_image_idx = (
    indices[n_val_images:]
)

print(
    f"Training images: "
    f"{len(train_image_idx)}"
)

for i in train_image_idx:
    print(
        " -",
        metadata[i]["name"]
    )

print(
    f"\nValidation images: "
    f"{len(val_image_idx)}"
)

for i in val_image_idx:
    print(
        " -",
        metadata[i]["name"]
    )

## 10. Balanced patch extraction

Positive patches are sampled around building pixels. Negative patches are sampled from regions containing almost no buildings.

In [ ]:
# ============================================================
# 10. BALANCED PATCH SAMPLING
# ============================================================

def sample_patches_balanced(
    img,
    mask,
    patch_size,
    n_patches,
    pos_fraction,
    min_pos_ratio,
    max_neg_ratio,
    seed
):
    rng = np.random.default_rng(seed)

    H, W = mask.shape
    ps = patch_size

    if H < ps or W < ps:

        return (
            np.empty(
                (0, ps, ps, 3),
                dtype=np.float32
            ),
            np.empty(
                (0, ps, ps, 1),
                dtype=np.float32
            )
        )

    building_y, building_x = (
        np.where(mask > 0)
    )

    X = []
    Y = []

    n_pos_target = int(
        round(
            n_patches
            * pos_fraction
        )
    )

    n_neg_target = (
        n_patches
        - n_pos_target
    )

    # --------------------------------------------------------
    # POSITIVE PATCHES
    # --------------------------------------------------------

    tries = 0
    max_tries = max(
        5000,
        n_patches * 500
    )

    while (
        len(X) < n_pos_target
        and tries < max_tries
        and len(building_x) > 0
    ):

        tries += 1

        j = rng.integers(
            0,
            len(building_x)
        )

        cx = int(
            building_x[j]
        )

        cy = int(
            building_y[j]
        )

        x0 = int(
            np.clip(
                cx - ps // 2,
                0,
                W - ps
            )
        )

        y0 = int(
            np.clip(
                cy - ps // 2,
                0,
                H - ps
            )
        )

        pm = mask[
            y0:y0 + ps,
            x0:x0 + ps
        ]

        if (
            pm.shape == (ps, ps)
            and pm.mean()
            >= min_pos_ratio
        ):

            X.append(
                img[
                    y0:y0 + ps,
                    x0:x0 + ps
                ]
            )

            Y.append(
                pm[..., None]
            )

    # --------------------------------------------------------
    # NEGATIVE PATCHES
    # --------------------------------------------------------

    neg_count = 0
    tries = 0

    while (
        neg_count < n_neg_target
        and tries < max_tries
    ):

        tries += 1

        x0 = int(
            rng.integers(
                0,
                W - ps + 1
            )
        )

        y0 = int(
            rng.integers(
                0,
                H - ps + 1
            )
        )

        pm = mask[
            y0:y0 + ps,
            x0:x0 + ps
        ]

        if (
            pm.shape == (ps, ps)
            and pm.mean()
            <= max_neg_ratio
        ):

            X.append(
                img[
                    y0:y0 + ps,
                    x0:x0 + ps
                ]
            )

            Y.append(
                pm[..., None]
            )

            neg_count += 1

    return (
        np.asarray(
            X,
            dtype=np.float32
        ),
        np.asarray(
            Y,
            dtype=np.float32
        )
    )


def patches_from_indices(
    selected_indices,
    seed_offset
):
    X_parts = []
    Y_parts = []

    for k, idx in enumerate(
        selected_indices
    ):

        xp, yp = (
            sample_patches_balanced(
                images[idx],
                masks[idx],
                patch_size=PATCH_SIZE,
                n_patches=N_PATCHES_PER_IMAGE,
                pos_fraction=POS_FRACTION,
                min_pos_ratio=MIN_POS_RATIO,
                max_neg_ratio=MAX_NEG_RATIO,
                seed=(
                    SEED
                    + seed_offset
                    + k
                )
            )
        )

        print(
            metadata[idx]["name"],
            "->",
            len(xp),
            "patches"
        )

        if len(xp) > 0:
            X_parts.append(xp)
            Y_parts.append(yp)

    if not X_parts:
        raise RuntimeError(
            "No patches could be generated."
        )

    return (
        np.concatenate(
            X_parts,
            axis=0
        ),
        np.concatenate(
            Y_parts,
            axis=0
        )
    )


print("TRAIN PATCHES\n")

X_train, Y_train = (
    patches_from_indices(
        train_image_idx,
        seed_offset=0
    )
)

print("\nVALIDATION PATCHES\n")

X_val, Y_val = (
    patches_from_indices(
        val_image_idx,
        seed_offset=10000
    )
)

print("\nFinal arrays:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val  :", X_val.shape)
print("Y_val  :", Y_val.shape)

print(
    "\nTrain building-pixel ratio:",
    float(Y_train.mean())
)

print(
    "Val building-pixel ratio:",
    float(Y_val.mean())
)

In [ ]:
# ============================================================
# 11. PATCH SANITY CHECK
# ============================================================

n_show = min(
    6,
    len(X_train)
)

indices_show = np.random.default_rng(
    SEED
).choice(
    len(X_train),
    size=n_show,
    replace=False
)

fig, axes = plt.subplots(
    2,
    n_show,
    figsize=(3 * n_show, 6)
)

for col, idx in enumerate(
    indices_show
):

    axes[0, col].imshow(
        X_train[idx]
    )

    axes[0, col].set_title(
        "RGB"
    )

    axes[1, col].imshow(
        Y_train[
            idx,
            ...,
            0
        ],
        cmap="gray"
    )

    axes[1, col].set_title(
        "Mask"
    )

    axes[0, col].axis("off")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

## 12. TensorFlow pipeline and augmentation

In [ ]:
# ============================================================
# 12. TF.DATA + AUGMENTATION
# ============================================================

def augment_pair(
    x,
    y
):
    if (
        tf.random.uniform(())
        > 0.5
    ):
        x = tf.image.flip_left_right(
            x
        )
        y = tf.image.flip_left_right(
            y
        )

    if (
        tf.random.uniform(())
        > 0.5
    ):
        x = tf.image.flip_up_down(
            x
        )
        y = tf.image.flip_up_down(
            y
        )

    k = tf.random.uniform(
        (),
        minval=0,
        maxval=4,
        dtype=tf.int32
    )

    x = tf.image.rot90(
        x,
        k
    )

    y = tf.image.rot90(
        y,
        k
    )

    x = tf.image.random_brightness(
        x,
        max_delta=0.08
    )

    x = tf.image.random_contrast(
        x,
        lower=0.85,
        upper=1.15
    )

    x = tf.clip_by_value(
        x,
        0.0,
        1.0
    )

    return x, y


def make_dataset(
    X,
    Y,
    training
):
    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (X, Y)
        )
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(X),
            seed=SEED,
            reshuffle_each_iteration=True
        )

        ds = ds.map(
            augment_pair,
            num_parallel_calls=(
                tf.data.AUTOTUNE
            )
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


train_ds = make_dataset(
    X_train,
    Y_train,
    training=True
)

val_ds = make_dataset(
    X_val,
    Y_val,
    training=False
)

print("TensorFlow datasets ready.")

## 13. U-Net

Loss = Binary Cross-Entropy + soft Dice loss.

Metrics:

- binary accuracy;
- building IoU at threshold 0.5 during training;
- soft Dice coefficient.

In [ ]:
# ============================================================
# 13. U-NET MODEL
# ============================================================

from tensorflow.keras import (
    layers,
    Model
)


def conv_block(
    x,
    filters,
    dropout=0.0
):

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(
        x
    )

    x = layers.Activation(
        "relu"
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(
        x
    )

    x = layers.Activation(
        "relu"
    )(x)

    if dropout > 0:
        x = layers.Dropout(
            dropout
        )(x)

    return x


def build_unet(
    input_shape=(
        PATCH_SIZE,
        PATCH_SIZE,
        3
    ),
    base=32
):

    inp = layers.Input(
        shape=input_shape
    )

    # Encoder
    c1 = conv_block(
        inp,
        base
    )
    p1 = layers.MaxPool2D()(
        c1
    )

    c2 = conv_block(
        p1,
        base * 2
    )
    p2 = layers.MaxPool2D()(
        c2
    )

    c3 = conv_block(
        p2,
        base * 4,
        dropout=0.10
    )
    p3 = layers.MaxPool2D()(
        c3
    )

    c4 = conv_block(
        p3,
        base * 8,
        dropout=0.20
    )
    p4 = layers.MaxPool2D()(
        c4
    )

    # Bottleneck
    b = conv_block(
        p4,
        base * 16,
        dropout=0.30
    )

    # Decoder
    u4 = layers.UpSampling2D()(
        b
    )

    u4 = layers.Concatenate()(
        [u4, c4]
    )

    c5 = conv_block(
        u4,
        base * 8,
        dropout=0.20
    )

    u3 = layers.UpSampling2D()(
        c5
    )

    u3 = layers.Concatenate()(
        [u3, c3]
    )

    c6 = conv_block(
        u3,
        base * 4,
        dropout=0.10
    )

    u2 = layers.UpSampling2D()(
        c6
    )

    u2 = layers.Concatenate()(
        [u2, c2]
    )

    c7 = conv_block(
        u2,
        base * 2
    )

    u1 = layers.UpSampling2D()(
        c7
    )

    u1 = layers.Concatenate()(
        [u1, c1]
    )

    c8 = conv_block(
        u1,
        base
    )

    out = layers.Conv2D(
        1,
        1,
        activation="sigmoid"
    )(c8)

    return Model(
        inp,
        out
    )


def dice_coef(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    yt = tf.reshape(
        y_true,
        [-1]
    )

    yp = tf.reshape(
        y_pred,
        [-1]
    )

    intersection = tf.reduce_sum(
        yt * yp
    )

    denominator = (
        tf.reduce_sum(yt)
        + tf.reduce_sum(yp)
    )

    return (
        2.0 * intersection
        + smooth
    ) / (
        denominator
        + smooth
    )


def bce_dice_loss(
    y_true,
    y_pred
):

    bce = (
        tf.keras.losses
        .binary_crossentropy(
            y_true,
            y_pred
        )
    )

    bce = tf.reduce_mean(
        bce
    )

    return (
        bce
        + (
            1.0
            - dice_coef(
                y_true,
                y_pred
            )
        )
    )


strategy = (
    tf.distribute
    .MirroredStrategy()
)

print(
    "Training replicas:",
    strategy.num_replicas_in_sync
)

with strategy.scope():

    model = build_unet()

    model.compile(

        optimizer=(
            tf.keras.optimizers.Adam(
                learning_rate=LR
            )
        ),

        loss=bce_dice_loss,

        metrics=[
            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.BinaryIoU(
                target_class_ids=[1],
                threshold=0.5,
                name="building_iou"
            ),

            dice_coef
        ]
    )


model.summary()

## 14. Train

The best checkpoint is selected by validation loss. Early stopping restores the best weights.

In [ ]:
# ============================================================
# 14. TRAIN
# ============================================================

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        patience=4,
        factor=0.5,
        min_lr=1e-6,
        verbose=1
    )
]


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# 15. TRAINING CURVES
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

axes[0].plot(
    history.history["loss"],
    label="Train"
)

axes[0].plot(
    history.history["val_loss"],
    label="Validation"
)

axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()


axes[1].plot(
    history.history["building_iou"],
    label="Train"
)

axes[1].plot(
    history.history["val_building_iou"],
    label="Validation"
)

axes[1].set_title(
    "Building IoU @ 0.5"
)

axes[1].set_xlabel("Epoch")
axes[1].legend()


axes[2].plot(
    history.history["dice_coef"],
    label="Train"
)

axes[2].plot(
    history.history["val_dice_coef"],
    label="Validation"
)

axes[2].set_title(
    "Soft Dice"
)

axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

## 16. Tune the binary probability threshold

The network returns a probability map. Instead of assuming `0.5` is optimal, we search several thresholds on the validation patches and select the threshold with the highest F1 score.

In [ ]:
# ============================================================
# 16. THRESHOLD SELECTION
# ============================================================

def binary_metrics(
    y_true,
    y_pred
):

    yt = (
        y_true > 0.5
    ).astype(
        np.uint8
    ).reshape(-1)

    yp = (
        y_pred > 0
    ).astype(
        np.uint8
    ).reshape(-1)

    tp = int(
        np.sum(
            (yt == 1)
            & (yp == 1)
        )
    )

    fp = int(
        np.sum(
            (yt == 0)
            & (yp == 1)
        )
    )

    fn = int(
        np.sum(
            (yt == 1)
            & (yp == 0)
        )
    )

    tn = int(
        np.sum(
            (yt == 0)
            & (yp == 0)
        )
    )

    precision = (
        tp
        / (
            tp
            + fp
            + 1e-8
        )
    )

    recall = (
        tp
        / (
            tp
            + fn
            + 1e-8
        )
    )

    f1 = (
        2
        * precision
        * recall
        / (
            precision
            + recall
            + 1e-8
        )
    )

    iou = (
        tp
        / (
            tp
            + fp
            + fn
            + 1e-8
        )
    )

    dice = (
        2 * tp
        / (
            2 * tp
            + fp
            + fn
            + 1e-8
        )
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision":
            float(precision),
        "recall":
            float(recall),
        "f1":
            float(f1),
        "iou":
            float(iou),
        "dice":
            float(dice)
    }


val_probs = model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

threshold_rows = []

for threshold in np.arange(
    0.20,
    0.81,
    0.05
):

    pred_bin = (
        val_probs
        >= threshold
    ).astype(
        np.uint8
    )

    metrics = binary_metrics(
        Y_val,
        pred_bin
    )

    threshold_rows.append({
        "threshold":
            float(
                round(
                    threshold,
                    2
                )
            ),
        **metrics
    })


threshold_df = pd.DataFrame(
    threshold_rows
)

display(
    threshold_df[
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "iou",
            "dice"
        ]
    ]
)


best_row = (
    threshold_df
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

best_threshold = float(
    best_row["threshold"]
)

best_metrics = {
    key: (
        int(best_row[key])
        if key
        in ["tp", "fp", "fn", "tn"]
        else float(best_row[key])
    )
    for key in [
        "tp",
        "fp",
        "fn",
        "tn",
        "precision",
        "recall",
        "f1",
        "iou",
        "dice"
    ]
}


print(
    "\n✅ Best threshold:",
    best_threshold
)

print(
    "Validation metrics:",
    best_metrics
)

In [ ]:
# ============================================================
# 17. QUALITATIVE VALIDATION PREDICTIONS
# ============================================================

n_show = min(
    6,
    len(X_val)
)

rng_vis = np.random.default_rng(
    SEED + 99
)

show_idx = rng_vis.choice(
    len(X_val),
    size=n_show,
    replace=False
)

fig, axes = plt.subplots(
    n_show,
    4,
    figsize=(16, 4 * n_show)
)

if n_show == 1:
    axes = np.expand_dims(
        axes,
        axis=0
    )

for row, idx in enumerate(
    show_idx
):

    rgb = X_val[idx]

    gt = (
        Y_val[
            idx,
            ...,
            0
        ]
        > 0.5
    )

    prob = val_probs[
        idx,
        ...,
        0
    ]

    pred = (
        prob
        >= best_threshold
    )

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title("RGB")

    axes[row, 1].imshow(
        gt,
        cmap="gray"
    )
    axes[row, 1].set_title(
        "Ground truth"
    )

    axes[row, 2].imshow(
        prob,
        cmap="viridis",
        vmin=0,
        vmax=1
    )
    axes[row, 2].set_title(
        "Probability"
    )

    axes[row, 3].imshow(rgb)
    axes[row, 3].imshow(
        pred,
        cmap="Reds",
        alpha=0.40
    )
    axes[row, 3].set_title(
        f"Prediction ≥ "
        f"{best_threshold:.2f}"
    )

    for col in range(4):
        axes[row, col].axis(
            "off"
        )

plt.tight_layout()
plt.show()

## 18. Save the model and inference configuration

The inference notebook must reproduce the same:

- RGB band choice;
- percentile normalization;
- patch size;
- probability threshold.

In [ ]:
# ============================================================
# 18. SAVE FINAL ARTIFACTS
# ============================================================

# Save once more to ensure the restored best weights
# are the final exported model.
model.save(
    MODEL_PATH
)

training_config = {

    "model_file":
        os.path.basename(
            MODEL_PATH
        ),

    "input_bands_1_based":
        BANDS,

    "band_meaning": {
        "1":
            "B04 / Red",
        "2":
            "B03 / Green",
        "3":
            "B02 / Blue"
    },

    "normalization": {
        "type":
            "per_image_per_band_percentile",
        "pmin":
            2,
        "pmax":
            98,
        "output_range":
            [0.0, 1.0]
    },

    "patch_size":
        PATCH_SIZE,

    "recommended_inference_stride":
        PATCH_SIZE // 2,

    "prediction_threshold":
        best_threshold,

    "validation_metrics_at_selected_threshold":
        best_metrics,

    "dataset_path":
        DATASET_PATH,

    "number_of_input_tifs":
        len(all_tif_files),

    "number_of_labelled_images":
        len(images),

    "number_of_training_images":
        len(train_image_idx),

    "number_of_validation_images":
        len(val_image_idx),

    "number_of_training_patches":
        int(len(X_train)),

    "number_of_validation_patches":
        int(len(X_val)),

    "label_source":
        "OpenStreetMap building footprints",

    "train_images": [
        metadata[i]["name"]
        for i in train_image_idx
    ],

    "validation_images": [
        metadata[i]["name"]
        for i in val_image_idx
    ],

    "seed":
        SEED
}


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        training_config,
        f,
        indent=2
    )


threshold_csv = os.path.join(
    OUTPUT_DIR,
    "threshold_search.csv"
)

threshold_df.to_csv(
    threshold_csv,
    index=False
)


print("✅ MODEL")
print(MODEL_PATH)

print("\n✅ CONFIG")
print(CONFIG_PATH)

print("\n✅ THRESHOLD TABLE")
print(threshold_csv)

In [ ]:
# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"\nLabelled images      : "
    f"{len(images)}"
)

print(
    f"Training images      : "
    f"{len(train_image_idx)}"
)

print(
    f"Validation images    : "
    f"{len(val_image_idx)}"
)

print(
    f"Training patches     : "
    f"{len(X_train)}"
)

print(
    f"Validation patches   : "
    f"{len(X_val)}"
)

print(
    f"Best threshold       : "
    f"{best_threshold:.2f}"
)

print(
    f"Validation F1        : "
    f"{best_metrics['f1']:.4f}"
)

print(
    f"Validation IoU       : "
    f"{best_metrics['iou']:.4f}"
)

print(
    f"Validation Dice      : "
    f"{best_metrics['dice']:.4f}"
)

print("\nFiles to download from Kaggle Output:")

print(
    "  unet_buildings.keras"
)

print(
    "  segmentation_training_config.json"
)

print(
    "  threshold_search.csv"
)

print(
    "\nThese are the inputs for the "
    "next inference notebook."
)

# Next pipeline stage

After this notebook finishes, **do not retrain the U-Net for each new user image**.

The operational pipeline becomes:

```text
Sentinel-2
    ↓
OpenSR
    ↓
new sr.tif
    ↓
load unet_buildings.keras
    ↓
same RGB normalization
    ↓
tiled U-Net inference
    ↓
apply saved probability threshold
    ↓
building_mask.tif
    ↓
PV estimation notebook
```